In [ ]:
# Import required libraries
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add parent directory to path for importing app modules
sys.path.insert(0, str(Path.cwd().parent))

# Import custom modules
from app.preprocessing.feature_engineering import FeatureEngineer, FeatureSelector, FeatureNormalizer

# Configure visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## Conclusion

This analysis demonstrates comprehensive feature engineering for healthcare anomaly detection:

✅ **Base Features**: 7 direct vital signs from clinical monitoring  
✅ **Derived Features**: 4 calculated physiological indicators  
✅ **Feature Selection**: Core physiological features identified for anomaly detection  
✅ **Independence**: Low inter-feature correlation ensures model robustness  
✅ **Variance**: Each feature contributes distinct information to detection  
✅ **Normalization**: Methods prepared for model training

### Key Features for Anomaly Detection:
1. **Heart Rate (HR)** - Direct rhythm/rate measurement
2. **Oxygen Saturation (SpO₂)** - Respiratory function
3. **Body Temperature** - Metabolic status
4. **Blood Pressure (Sys/Dia)** - Cardiovascular mechanics
5. **Heart Rate Variability (HRV)** - Autonomic function
6. **Mean Arterial Pressure (MAP)** - Perfusion adequacy

These 7 features collectively represent comprehensive physiological monitoring across cardiovascular, respiratory, and autonomic systems - essential for robust health anomaly detection.

In [ ]:
# Load sample healthcare data
# Check for data file in various possible locations
data_paths = [
    'data/healthcare_data.csv',
    '../data/healthcare_data.csv',
    '../../data/healthcare_data.csv',
]

data_file = None
for path in data_paths:
    if os.path.exists(path):
        data_file = path
        break

if data_file:
    df = pd.read_csv(data_file)
    print(f"✓ Data loaded from: {data_file}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
else:
    print("⚠ Data file not found. Creating sample data for demonstration...")
    # Create sample healthcare data for demo
    np.random.seed(42)
    n_samples = 1000
    df = pd.DataFrame({
        'HR': np.random.normal(75, 10, n_samples),
        'SpO2': np.random.normal(97, 2, n_samples),
        'Temperature': np.random.normal(37, 0.5, n_samples),
        'SysBP': np.random.normal(120, 15, n_samples),
        'DiaBP': np.random.normal(80, 10, n_samples),
        'Glucose': np.random.normal(100, 20, n_samples),
        'Cholesterol': np.random.normal(200, 30, n_samples),
        'Age': np.random.randint(20, 80, n_samples),
    })
    print(f"✓ Sample data created: {df.shape}")

# Initialize Feature Engineer
fe = FeatureEngineer()
print("\n✓ FeatureEngineer initialized")

In [ ]:
## Feature Extraction and Engineering

# Extract base features
base_features = df[['HR', 'SpO2', 'Temperature', 'SysBP', 'DiaBP', 'Glucose', 'Cholesterol']].copy()
base_features.columns = ['HR', 'SpO2', 'Temp', 'SysBP', 'DiaBP', 'Glucose', 'Cholesterol']

# Extract derived features using FeatureEngineer
derived_features_dict = fe.extract_derived_features(df)
all_features = fe.extract_all_features(df)

# Create core features dataframe (7 essential features)
core_features = base_features[['HR', 'SpO2', 'Temp', 'SysBP', 'DiaBP']].copy()
core_features['HRV'] = derived_features_dict.get('HRV', np.zeros(len(df)))
core_features['MAP'] = derived_features_dict.get('MAP', np.zeros(len(df)))

print("✓ Feature Extraction Complete")
print(f"  Base Features Shape: {base_features.shape}")
print(f"  All Features Shape: {all_features.shape}")
print(f"  Core Features Shape: {core_features.shape}")
print(f"\nCore Features:")
print(core_features.head())

## Section 8: Feature Summary Report

In [ ]:
## Feature Selection Based on Variance and Correlation

# Initialize Feature Selector
fs = FeatureSelector()
selected_features = fs.select_core_features(core_features)

print("✓ Core Features Selected:")
print(f"  Features: {selected_features}")
print(f"  Count: {len(selected_features)}")

# Calculate correlation matrix
correlation_matrix = core_features.corr()

# Find high correlation pairs
high_corr_pairs = []
corr_threshold = 0.7
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > corr_threshold:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

print(f"\n✓ Correlation Analysis:")
print(f"  High correlation pairs (|r| > {corr_threshold}): {len(high_corr_pairs)}")

# Calculate variance statistics
variance_stats = {}
for col in core_features.columns:
    data = core_features[col].dropna()
    variance_stats[col] = {
        'variance': data.var(),
        'std': data.std(),
        'mean': data.mean(),
        'count': len(data)
    }

# Sort by variance
sorted_variance = sorted(variance_stats.items(), key=lambda x: x[1]['variance'], reverse=True)

print(f"\n✓ Variance Analysis:")
for feature, stats in sorted_variance[:7]:
    print(f"  {feature}: {stats['variance']:.2f}")

In [ ]:
## Feature Normalization using StandardScaler

# Initialize normalizer with StandardScaler
normalizer = FeatureNormalizer()

# Fit normalizer on training data
normalizer.fit(core_features)

# Get scaling parameters (mean and std for each feature)
scaling_params = normalizer.get_scaling_params()

print("✓ StandardScaler Normalization")
print("\nScaling Parameters (Mean ± Std):")
for feature, params in scaling_params.items():
    print(f"  {feature:10}: Mean={params['mean']:8.2f}, Std={params['std']:8.2f}")

# Transform core features using StandardScaler
normalized_features = normalizer.transform(core_features)

print(f"\n✓ Normalization Complete")
print(f"  Original Features Shape: {core_features.shape}")
print(f"  Normalized Features Shape: {normalized_features.shape}")

# Verify normalization (should be mean ≈ 0, std ≈ 1)
print(f"\nNormalized Features Statistics:")
print(f"  Means: {normalized_features.mean().round(6).to_dict()}")
print(f"  Stds: {normalized_features.std().round(4).to_dict()}")

# Display samples
print(f"\nOriginal Features (first 5 rows):")
print(core_features.head())
print(f"\nNormalized Features using StandardScaler (first 5 rows):")
print(normalized_features.head())

In [ ]:
## Save Scaler for Production Use

# Create models directory if it doesn't exist
import os
scaler_dir = '../models/scalers'
os.makedirs(scaler_dir, exist_ok=True)

# Save the fitted scaler
scaler_path = os.path.join(scaler_dir, 'feature_scaler.joblib')
normalizer.save(scaler_path)

print(f"✓ Scaler saved to: {scaler_path}")

# Demonstrate loading and applying scaler
print("\nTesting Scaler Persistence:")
normalizer_loaded = FeatureNormalizer()
normalizer_loaded.load(scaler_path)

# Apply loaded scaler on same features
test_features = normalizer_loaded.transform(core_features[:10])
print(f"✓ Loaded scaler applied successfully")
print(f"  Transformed 10 samples: {test_features.shape}")

In [ ]:
## Visualization: Original vs StandardScaler Normalized Features

# Create comparison visualization
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

features_to_plot = core_features.columns[:8]

for idx, feature in enumerate(features_to_plot):
    # Original data histogram
    axes[idx].hist(core_features[feature].dropna(), bins=30, alpha=0.6, 
                   color='steelblue', label='Original', edgecolor='black')
    
    # Normalized data histogram
    ax2 = axes[idx].twinx()
    ax2.hist(normalized_features[feature].dropna(), bins=30, alpha=0.6, 
             color='coral', label='StandardScaler Normalized', edgecolor='black')
    
    axes[idx].set_title(f'{feature} - Original vs Normalized', fontweight='bold', fontsize=10)
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency (Original)', color='steelblue', fontweight='bold')
    ax2.set_ylabel('Frequency (Normalized)', color='coral', fontweight='bold')
    axes[idx].tick_params(axis='y', labelcolor='steelblue')
    ax2.tick_params(axis='y', labelcolor='coral')
    axes[idx].legend(loc='upper left', fontsize=8)
    ax2.legend(loc='upper right', fontsize=8)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../plots/13_standardscaler_normalization.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ StandardScaler normalization comparison visualization saved")

## Section 7: Feature Normalization

In [ ]:
# Feature variance analysis
print("=" * 80)
print("FEATURE VARIANCE ANALYSIS")
print("=" * 80)

feature_variance = {}
for feature in core_features.columns:
    data = core_features[feature].dropna()
    variance = data.var()
    std_coeff = data.std() / data.mean() if data.mean() != 0 else 0
    feature_variance[feature] = {
        'variance': variance,
        'std': data.std(),
        'cv': std_coeff  # Coefficient of variation
    }

# Sort by variance
sorted_variance = sorted(feature_variance.items(), key=lambda x: x[1]['variance'], reverse=True)

print("\nFeatures ranked by variance:")
for feature, stats in sorted_variance:
    print(f"  {feature:10}: Variance={stats['variance']:10.2f}, CV={stats['cv']:6.3f}")

# Visualize variance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Variance bar plot
features_sorted = [f[0] for f in sorted_variance]
variances = [f[1]['variance'] for f in sorted_variance]
axes[0].barh(features_sorted, variances, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Variance', fontweight='bold')
axes[0].set_title('Feature Variance Ranking', fontweight='bold', fontsize=12)
axes[0].grid(alpha=0.3, axis='x')

# Coefficient of variation
cvs = [f[1]['cv'] for f in sorted_variance]
axes[1].barh(features_sorted, cvs, color='coral', alpha=0.7)
axes[1].set_xlabel('Coefficient of Variation', fontweight='bold')
axes[1].set_title('Feature Variability (CV)', fontweight='bold', fontsize=12)
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('../plots/12_feature_variance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Variance analysis visualization saved")

## Section 6: Feature Importance & Variance Analysis

In [ ]:
# Correlation analysis
print("=" * 80)
print("FEATURE CORRELATION ANALYSIS")
print("=" * 80)

# Get correlation for core features
core_features_only = core_features.dropna()
correlation_matrix = core_features_only.corr()

print("\nCorrelation Matrix (Core Features):")
print(correlation_matrix.round(3))

# Find high correlations
print("\n\nHigh Correlations (|r| > 0.7):")
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr = correlation_matrix.iloc[i, j]
        if abs(corr) > 0.7:
            col1 = correlation_matrix.columns[i]
            col2 = correlation_matrix.columns[j]
            high_corr_pairs.append((col1, col2, corr))
            print(f"  {col1:10} <-> {col2:10}: {corr:7.3f}")

if not high_corr_pairs:
    print("  No high correlations found - features are well-distributed")

# Visualize correlation
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Core Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/11_feature_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Correlation visualization saved")

## Section 5: Feature Correlation Analysis

In [ ]:
# Initialize feature selector
fs = FeatureSelector(all_features)

print("=" * 80)
print("FEATURE SELECTION")
print("=" * 80)

# Select core physiological features
core_features = fs.select_core_features()
print(f"\nCore Physiological Features ({len(fs.selected_features)}):")
print(f"  {', '.join(fs.selected_features)}")

core_stats = fs.get_feature_stats()
print("\nCore Feature Statistics:")
for feature, stats in core_stats.items():
    print(f"\n{feature}:")
    print(f"  Count: {stats['count']}")
    print(f"  Mean: {stats['mean']:.2f} ± {stats['std']:.2f}")
    print(f"  Range: [{stats['min']:.2f}, {stats['max']:.2f}]")
    print(f"  Missing: {stats['missing']} ({stats['missing_pct']:.1f}%)")

## Section 4: Feature Selection for Anomaly Detection

In [ ]:
# Detailed analysis of derived features
print("=" * 80)
print("DERIVED FEATURES ANALYSIS")
print("=" * 80)

derived_features = ['MAP', 'HRV', 'PP', 'RPP', 'SpO2_deviation', 'Temp_deviation']

for feature in derived_features:
    if feature in all_features.columns:
        data = all_features[feature].dropna()
        print(f"\n{feature}")
        print(f"  Count: {len(data)}")
        print(f"  Mean: {data.mean():.2f}")
        print(f"  Std Dev: {data.std():.2f}")
        print(f"  Min: {data.min():.2f}")
        print(f"  Max: {data.max():.2f}")
        print(f"  Median: {data.median():.2f}")
        print(f"  Missing: {all_features[feature].isnull().sum()} ({100*all_features[feature].isnull().sum()/len(all_features):.1f}%)")

In [ ]:
# Visualize derived features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, feature in enumerate(derived_features):
    if feature in all_features.columns:
        data = all_features[feature].dropna()
        
        axes[idx].hist(data, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
        axes[idx].axvline(data.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {data.mean():.2f}')
        axes[idx].axvline(data.median(), color='green', linestyle='--', linewidth=2, label=f'Median: {data.median():.2f}')
        axes[idx].set_title(f'{feature} Distribution', fontweight='bold', fontsize=11)
        axes[idx].set_xlabel('Value')
        axes[idx].set_ylabel('Frequency')
        axes[idx].legend(fontsize=9)
        axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../plots/10_derived_features.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Derived features visualization saved")

## Section 3: Derived Feature Calculations

### MAP (Mean Arterial Pressure)
Formula: MAP = (Systolic + 2×Diastolic) / 3
- Normal range: 70-100 mmHg
- Indicates average perfusion pressure

### HRV (Heart Rate Variability)
- Calculated as rolling standard deviation of heart rate
- Window: 5 consecutive readings
- Indicator of autonomic nervous system function

### RPP (Rate Pressure Product)
Formula: RPP = Heart Rate × Systolic BP
- Surrogate for myocardial oxygen demand
- Higher values indicate greater cardiac work

In [ ]:
# Extract features
all_features = fe.get_anomaly_features()

print("=" * 80)
print("FEATURE EXTRACTION COMPLETE")
print("=" * 80)
print(f"\nTotal features extracted: {all_features.shape[1]}")
print(f"Feature columns: {all_features.columns.tolist()}\n")

# Display feature summary
print("Feature Statistics:")
for col in all_features.columns[:8]:
    print(f"\n{col}:")
    data = all_features[col].dropna()
    print(f"  Mean: {data.mean():.2f} ± {data.std():.2f}")
    print(f"  Range: [{data.min():.2f}, {data.max():.2f}]")
    print(f"  Missing: {all_features[col].isnull().sum()}")

## Section 2: Extract Base & Derived Features

# Feature Engineering & Selection for Anomaly Detection

This notebook demonstrates extraction and selection of physiological features for robust anomaly detection.

## Features Extracted

### Base Features (Direct from Dataset)
- **Heart Rate (HR)** - Cardiovascular function indicator
- **Oxygen Saturation (SpO₂)** - Respiratory adequacy
- **Body Temperature** - Metabolic status
- **Systolic & Diastolic BP** - Cardiovascular mechanics
- **Blood Glucose** - Metabolic health
- **Cholesterol** - Lipid metabolism

### Derived Features (Calculated)
- **Heart Rate Variability (HRV)** - Autonomic nervous system function
- **Mean Arterial Pressure (MAP)** - Average perfusion pressure
- **Pulse Pressure (PP)** - Arterial stiffness indicator
- **Rate Pressure Product (RPP)** - Myocardial oxygen demand
- **SpO₂ & Temperature Deviations** - Distance from normal